In [28]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

In [29]:
class BatteryDataset(Dataset):
    def __init__(self, folder_path, seq_len=64):
        self.seq_len = seq_len
        self.samples = []
 
        files = [os.path.join(folder_path, f)
                 for f in os.listdir(folder_path) if f.endswith(".csv")]
 
        self.features = [
            'Voltage_measured',
            'Current_measured',
            'Temperature_measured',
            'Current_charge',
            'Voltage_charge',
            'Time_uniform',
            'cycle_index',
            'mode_flag'
        ]
 
        all_cycles = [int(re.findall(r'\d+', f)[-1]) for f in files]
        max_cycle = max(all_cycles)
 
        capacities = []
        for file in files:
            df = pd.read_csv(file).ffill().fillna(0)
            if 'Current_measured' in df.columns:
                discharge = df[df['Current_measured'] < 0]
                if len(discharge) > 0:
                    Q = np.sum(np.abs(discharge['Current_measured'])) / 3600
                    capacities.append(Q)
 
        initial_capacity = max(capacities) if capacities else 1.0
 
        # ── Global stats pass (fix: compute across ALL files, not per-file) ──
        all_feature_data = []
        valid_files = []
 
        for file in files:
            df = pd.read_csv(file)
            df.columns = df.columns.str.strip()
            df = df.ffill().fillna(0)
 
            if 'SOC' not in df.columns:
                continue
 
            cycle_num = int(re.findall(r'\d+', file)[-1])
            df['cycle_index'] = cycle_num / max_cycle
            df['mode_flag'] = (df['Current_measured'] < 0).astype(int)
 
            for col in self.features:
                if col not in df.columns:
                    df[col] = 0
 
            df = df[self.features + ['SOC']]
            df[self.features] = df[self.features].apply(
                pd.to_numeric, errors='coerce').fillna(0)
 
            if len(df) >= seq_len:
                all_feature_data.append(df[self.features].values)
                valid_files.append(file)
 
        # Global mean/std across all training data
        combined = np.vstack(all_feature_data)
        self.global_mean = combined.mean(axis=0)
        self.global_std  = combined.std(axis=0) + 1e-8
 
        # ── Main sample-building loop ──────────────────────────────────────
        for file in valid_files:
            df = pd.read_csv(file)
            df.columns = df.columns.str.strip()
            df = df.ffill().fillna(0)
 
            cycle_num = int(re.findall(r'\d+', file)[-1])
            df['cycle_index'] = cycle_num / max_cycle
            df['mode_flag'] = (df['Current_measured'] < 0).astype(int)
 
            for col in self.features:
                if col not in df.columns:
                    df[col] = 0
 
            df = df[self.features + ['SOC']]
            df[self.features] = df[self.features].apply(
                pd.to_numeric, errors='coerce').fillna(0)
 
            data = df[self.features].values
            soc  = df['SOC'].values
            temp = df['Temperature_measured'].values   # real, not normalised
 
            discharge = df[df['Current_measured'] < 0]
            Q_cycle = (np.sum(np.abs(discharge['Current_measured'])) / 3600
                       if len(discharge) > 0 else initial_capacity)
            soh_val = Q_cycle / initial_capacity
            soh = np.full_like(soc, soh_val)
 
            # Apply global normalisation (inputs only)
            data = (data - self.global_mean) / self.global_std
 
            # Stride = 16 → reduces overlap between adjacent windows
            stride = 16
            for i in range(0, len(df) - seq_len, stride):
                X = data[i:i + seq_len]
                y = np.array([
                    soc[i + seq_len],
                    soh[i + seq_len],
                    temp[i + seq_len]
                ])
                if X.shape[1] != len(self.features):
                    continue
                self.samples.append((X, y))
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        X, y = self.samples[idx]
        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

In [30]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
 
    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        scores = self.score(lstm_out)          # (batch, seq_len, 1)
        weights = torch.softmax(scores, dim=1) # (batch, seq_len, 1)
        context = (weights * lstm_out).sum(dim=1)  # (batch, hidden_dim)
        return context, weights.squeeze(-1)    # also return weights for viz
 
 
class PredictorAgent(nn.Module):
 
    def __init__(self, input_size, hidden_dim=64, num_layers=2, dropout=0.4):
        super().__init__()
 
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,          # between LSTM layers
            bidirectional=True
        )
 
        # BiLSTM doubles the hidden dim (forward + backward)
        bilstm_out_dim = hidden_dim * 2
 
        self.attention = AdditiveAttention(bilstm_out_dim)
 
        self.dropout = nn.Dropout(0.3)
 
        self.fc = nn.Sequential(
            nn.Linear(bilstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)          # [SoC, SoH, Temperature]
        )
 
    def forward(self, x):
        lstm_out, _ = self.bilstm(x)         # (batch, seq, hidden*2)
        context, attn_weights = self.attention(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, attn_weights

In [31]:
def battery_level_split(folder_path, train_ratio=0.8):
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])
    n_train = int(len(files) * train_ratio)
    train_files = files[:n_train]
    test_files  = files[n_train:]
    return train_files, test_files

In [32]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        preds, _ = model(X)

        loss_soc  = nn.MSELoss()(preds[:, 0], y[:, 0])
        loss_soh  = nn.MSELoss()(preds[:, 1], y[:, 1])
        loss_temp = nn.MSELoss()(preds[:, 2], y[:, 2])

        loss = loss_soc + 0.5 * loss_soh + 0.5 * loss_temp

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [33]:
def evaluate_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds, _ = model(X)
            loss = nn.MSELoss()(preds, y)
            total_loss += loss.item()

    return total_loss / len(loader)

In [34]:
def predict_with_confidence(model, sequence, n_passes=30, device=None):
    if device is None:
        device = next(model.parameters()).device
 
    model.train()   # keep dropout active for MC sampling
 
    if not isinstance(sequence, torch.Tensor):
        sequence = torch.tensor(sequence, dtype=torch.float32)
    sequence = sequence.unsqueeze(0).to(device)   # (1, seq_len, features)
 
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            pred, _ = model(sequence)
            preds.append(pred.cpu().numpy())
 
    preds = np.array(preds)          # (n_passes, 1, 3)
    mean  = preds.mean(axis=0)[0]   # (3,)
    std   = preds.std(axis=0)[0]    # (3,)
 
    return {
        "soc":         float(mean[0]),
        "soh":         float(mean[1]),
        "temperature": float(mean[2]),
        "confidence":  float(np.mean(std))   # scalar uncertainty
    }

In [35]:
FOLDER_PATH = "/kaggle/input/datasets/arujatiwary/nasa-li-ion-dataset/ECM_processed_cycles"
SEQ_LEN     = 64
BATCH_SIZE  = 64
EPOCHS      = 30
LR          = 3e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print("Building dataset...")
dataset = BatteryDataset(FOLDER_PATH, seq_len=SEQ_LEN)
print(f"Total samples: {len(dataset)}")

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size
train_ds, test_ds = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

sample_X, _ = dataset[0]
input_size = sample_X.shape[1]

model = PredictorAgent(input_size=input_size).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=4, factor=0.5)

best_test_loss = float("inf")
patience_counter = 0
EARLY_STOP_PATIENCE = 8

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    test_loss  = evaluate_epoch(model, test_loader, device)
    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']
    gap = test_loss - train_loss
    print(f"Epoch {epoch+1:2d} | Train {train_loss:.4f} | Test {test_loss:.4f} | Gap {gap:.4f} | LR {current_lr:.2e}")

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_predictor.pth")
        print("  → saved best model")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

Device: cuda
Building dataset...
Total samples: 165718
Trainable parameters: 166,340
Epoch  1 | Train 0.9445 | Test 0.0521 | Gap -0.8924 | LR 3.00e-04
  → saved best model
Epoch  2 | Train 0.1616 | Test 0.0450 | Gap -0.1166 | LR 3.00e-04
  → saved best model
Epoch  3 | Train 0.1107 | Test 0.0440 | Gap -0.0667 | LR 3.00e-04
  → saved best model
Epoch  4 | Train 0.0933 | Test 0.0436 | Gap -0.0497 | LR 3.00e-04
  → saved best model
Epoch  5 | Train 0.0864 | Test 0.0430 | Gap -0.0433 | LR 3.00e-04
  → saved best model
Epoch  6 | Train 0.0817 | Test 0.0462 | Gap -0.0355 | LR 3.00e-04
Epoch  7 | Train 0.0794 | Test 0.0419 | Gap -0.0376 | LR 3.00e-04
  → saved best model
Epoch  8 | Train 0.0775 | Test 0.0411 | Gap -0.0364 | LR 3.00e-04
  → saved best model
Epoch  9 | Train 0.0758 | Test 0.0419 | Gap -0.0340 | LR 3.00e-04
Epoch 10 | Train 0.0742 | Test 0.0425 | Gap -0.0317 | LR 3.00e-04
Epoch 11 | Train 0.0733 | Test 0.0444 | Gap -0.0290 | LR 3.00e-04
Epoch 12 | Train 0.0726 | Test 0.0435 | Ga

In [36]:
model.load_state_dict(torch.load("best_predictor.pth"))

sample_X, sample_y = dataset[0]
result = predict_with_confidence(model, sample_X, n_passes=30, device=device)

print("Prediction:", result)
print("Ground truth → SoC: {:.4f}, SoH: {:.4f}, Temp: {:.2f}".format(
    float(sample_y[0]), float(sample_y[1]), float(sample_y[2])
))

Prediction: {'soc': 0.06747661530971527, 'soh': 0.31067517399787903, 'temperature': 6.694737434387207, 'confidence': 0.06497877836227417}
Ground truth → SoC: 0.0133, SoH: 0.0000, Temp: 6.71


In [37]:
model.eval()
with torch.no_grad():
    X_t = sample_X.unsqueeze(0).to(device)
    _, attn = model(X_t)
    weights = attn.cpu().numpy()[0]

top5 = np.argsort(weights)[-5:][::-1]
print("Top-5 most attended timesteps:")
for rank, t in enumerate(top5):
    print(f"  #{rank+1}: timestep {t:2d}  weight={weights[t]:.4f}")

Top-5 most attended timesteps:
  #1: timestep 62  weight=0.0934
  #2: timestep 61  weight=0.0929
  #3: timestep 60  weight=0.0886
  #4: timestep 63  weight=0.0885
  #5: timestep 59  weight=0.0820
